In [1]:
# ── Fine-tuning LaBSE for sentiment classification — Colab GPU ──
# Companion to finetune_labse_grammar_of_luxury.ipynb, which did the same for
# aspect. Everything is held identical to that run except the head: one logit
# instead of four. The comparison isolates the task, not the setup.
# All imports and setup in one cell.

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

# ── Core ──
import os, random
import numpy as np
import pandas as pd

# ── Deep Learning ──
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Transformers ──
import transformers
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

# ── Metrics ──
from sklearn.metrics import f1_score

# ── Reproducibility (same seed as the frozen heads and the aspect fine-tune) ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Setup done. Device:", DEVICE)
print("torch", torch.__version__, "| transformers", transformers.__version__)

CUDA available: True
GPU: Tesla T4
Setup done. Device: cuda
torch 2.11.0+cu128 | transformers 5.13.1


In [2]:
# ── Upload the exact split exported by the main notebook ──
# Same rows, same SEED as the frozen heads and the aspect fine-tune, so all
# three models are scored on identical data. The files already carry a
# `sentiment` column (1 = pos, 0 = neg) — no new export was needed.

from google.colab import files
print("Select finetune_train.csv, finetune_val.csv and finetune_test.csv")
uploaded = files.upload()

train_df = pd.read_csv("finetune_train.csv")
val_df   = pd.read_csv("finetune_val.csv")
test_df  = pd.read_csv("finetune_test.csv")

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    n_pos = int(df["sentiment"].sum())
    print(f"{name:6s} {len(df):5d} rows   positive {n_pos:4d} "
          f"({n_pos/len(df):.0%})   negative {len(df)-n_pos:4d}")


Select finetune_train.csv, finetune_val.csv and finetune_test.csv


Saving finetune_test.csv to finetune_test.csv
Saving finetune_train.csv to finetune_train.csv
Saving finetune_val.csv to finetune_val.csv
train   1192 rows   positive  897 (75%)   negative  295
val      255 rows   positive  190 (75%)   negative   65
test     256 rows   positive  194 (76%)   negative   62


In [3]:
# ── Dataset, tokenizer, model ──
# Held identical to the aspect fine-tune: same encoder, same MAX_LEN of 128,
# same batch size. The only difference is the head — one logit, not four —
# so any difference in outcome is attributable to the task, not the setup.

MODEL_NAME = "sentence-transformers/LaBSE"
MAX_LEN    = 128
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, df):
        self.texts  = df["text"].astype(str).tolist()
        self.labels = df["sentiment"].astype("float32").values

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, padding="max_length",
                        max_length=MAX_LEN, return_tensors="pt")
        return {"input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels":         torch.tensor(self.labels[i])}

train_loader = DataLoader(SentimentDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SentimentDataset(val_df),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(SentimentDataset(test_df),  batch_size=BATCH_SIZE)

class LaBSESentiment(nn.Module):
    """LaBSE with a single classification logit on the pooled [CLS] token."""
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.1)
        self.head    = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        return self.head(self.dropout(cls)).squeeze(-1)

model = LaBSESentiment().to(DEVICE)

# Class weighting, as in the frozen head: negatives are the minority, so the
# common positive class is down-weighted. 295/897 = 0.33 — the same value the
# frozen sentiment head used, derived from the same training split.
pos_weight = torch.tensor(train_df["sentiment"].eq(0).sum() / train_df["sentiment"].sum(),
                          dtype=torch.float32, device=DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

n_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_params:,}")
print(f"pos_weight: {pos_weight.item():.3f}")

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.88GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Trainable parameters: 470,927,617
pos_weight: 0.329


In [4]:
# ── Fine-tune: four epochs, checkpoint on best validation macro-F1 ──
# Identical schedule to the aspect fine-tune: AdamW at 2e-5, linear warmup,
# gradient clipping at 1.0, four epochs. Nothing here was tuned for sentiment;
# the point is to change one variable — the task — and hold the rest fixed.

EPOCHS = 4
LR     = 2e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

def evaluate(loader):
    model.eval()
    P, T = [], []
    with torch.no_grad():
        for b in loader:
            logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
            P.append((torch.sigmoid(logits) >= 0.5).int().cpu())
            T.append(b["labels"].int())
    P, T = torch.cat(P).numpy(), torch.cat(T).numpy()
    return f1_score(T, P, average="macro", zero_division=0), P, T

best_f1, best_state = -1, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for b in train_loader:
        optimizer.zero_grad()
        logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        loss = loss_fn(logits, b["labels"].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        losses.append(loss.item())
    val_f1, _, _ = evaluate(val_loader)
    print(f"epoch {epoch}  train_loss {np.mean(losses):.4f}  val macro-F1 {val_f1:.3f}")
    if val_f1 > best_f1:
        best_f1, best_state = val_f1, {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest val macro-F1: {best_f1:.3f}   (frozen head was 0.873 on val)")
model.load_state_dict(best_state)

epoch 1  train_loss 0.2249  val macro-F1 0.846
epoch 2  train_loss 0.1223  val macro-F1 0.881
epoch 3  train_loss 0.0694  val macro-F1 0.855
epoch 4  train_loss 0.0424  val macro-F1 0.857

Best val macro-F1: 0.881   (frozen head was 0.873 on val)


<All keys matched successfully>

In [5]:
# ── The honest number: the fine-tuned model on the held-out test set ──
# Same 256 reviews the frozen head was scored on, same 0.5 threshold, no tuning.

test_f1, P_test, T_test = evaluate(test_loader)

print(f"Fine-tuned sentiment — test macro-F1 {test_f1:.3f}   "
      f"(frozen head was 0.851)")
print(f"Gain over frozen: {test_f1 - 0.851:+.3f}\n")

acc = (P_test == T_test).mean()
print(f"Accuracy {acc:.3f}   (frozen head was 0.887)\n")

print("By language:\n")
print(f"  {'lang':5s} {'macro-F1':>9s} {'neg in test':>12s} {'n':>5s}")
for lang in ["en", "es", "pt", "it"]:
    m = (test_df["lang"] == lang).values
    n_neg = int((T_test[m] == 0).sum())
    f1 = f1_score(T_test[m], P_test[m], average="macro", zero_division=0)
    flag = "   (too few negatives — unreliable)" if n_neg < 10 else ""
    print(f"  {lang:5s} {f1:9.3f} {n_neg:12d} {m.sum():5d}{flag}")


Fine-tuned sentiment — test macro-F1 0.899   (frozen head was 0.851)
Gain over frozen: +0.048

Accuracy 0.926   (frozen head was 0.887)

By language:

  lang   macro-F1  neg in test     n
  en        0.942           30    70
  es        0.871           17    59
  pt        0.689            3    69   (too few negatives — unreliable)
  it        0.851           12    58
